# Week 3 — Rank, Null space, Projection, Least Squares และ Condition number

## ไฟล์ข้อมูลของสัปดาห์นี้ (ทั้งหมดอยู่ใน `data/week03/`)
- `biz_customers.csv` — ลูกค้า 1203 คน
- `biz_campaigns.csv` — แคมเปญ 240 รายการ
- `pb03_M.csv` — เมทริกซ์จำนวนเต็ม 14×18 (ตารางปริมาณวัตถุดิบ 14 สูตร x 18 ชนิด)

## กติกาที่ต้องระวัง

1. **แถวแรกของ `pb03_M.csv` คือชื่อคอลัมน์** อ่านด้วย `pd.read_csv(...).to_numpy()` ได้ 14×18 พอดี
2. **ดัชนีนับจาก 0** — ทั้ง "ลูกค้าแถวที่ r", "แคมเปญแถวที่ r" และ "แถวที่ 0 ถึง N"
3. **design matrix ที่มี intercept = คอลัมน์ 1 เป็นคอลัมน์แรก** แล้วตามด้วยตัวแปรตาม**ลำดับที่โจทย์เขียน**
4. **ใช้เฉพาะแถวที่โจทย์บอก** เช่น 891 แถวแรก = `.iloc[:891]` (ไม่ใช่การสุ่ม)
5. สูตรสำคัญ
   - Least Squares: β = (XᵀX)⁻¹Xᵀy → ใช้ `np.linalg.solve(X.T@X, X.T@y)`
   - Projection: P = A(AᵀA)⁻¹Aᵀ
   - Condition number (2-norm) = σ_max / σ_min = `np.linalg.cond(X, 2)`

> ทุก cell ข้อในไฟล์นี้**เป็นอิสระ** คัดลอกไปรันเดี่ยว ๆ ได้เลย
> (มี import และโหลดไฟล์ของตัวเองครบ) อ่านข้อไหนก็ดูข้อนั้นได้เลย

In [1]:
# --- ดาวน์โหลดไฟล์ข้อมูล (รันครั้งเดียว ถ้าไฟล์ยังไม่มี) ---
# ไฟล์ CSV ฝังอยู่ในหน้าเว็บเป็น base64 ในตัวแปร JS ชื่อ DATA{}
# ไฟล์ที่ใช้ของสัปดาห์ 3: biz_campaigns.csv, biz_customers.csv, pb03_M.csv
# cell นี้ไม่จำเป็นต่อการทำข้อ - ทุก cell ข้ออ่านไฟล์เองอยู่แล้ว
from pathlib import Path
import base64, re, urllib.request

PAGE_URL = 'https://popotaroman-cloud.github.io/mathsForDSUBU/practice/practice_all.html'
DATA_DIR = Path(f'data/week03')
NEEDED = ['biz_campaigns.csv',
          'biz_customers.csv',
          'pb03_M.csv']

missing = [n for n in NEEDED if not (DATA_DIR / n).exists()]
if missing:
    print('ดาวน์โหลด:', missing)
    html = urllib.request.urlopen(PAGE_URL, timeout=60).read().decode('utf-8')
    block = re.search(r'DATA=\{([^}]*)\}', html).group(1)
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    for n in missing:
        b64 = re.search(r'"' + re.escape(n) + r'"\s*:\s*"([A-Za-z0-9+/=]+)"', block).group(1)
        (DATA_DIR / n).write_bytes(base64.b64decode(b64))
        print('  บันทึก', DATA_DIR / n)
else:
    print('ไฟล์ครบแล้ว:', DATA_DIR)

for n in NEEDED:
    print(f'  {n:<22} {len(open(DATA_DIR / n, encoding="utf-8").read().splitlines()) - 1:>5} แถวข้อมูล')

ไฟล์ครบแล้ว: data\week03
  biz_campaigns.csv        240 แถวข้อมูล
  biz_customers.csv       1203 แถวข้อมูล
  pb03_M.csv                14 แถวข้อมูล


In [2]:
# --- ตรวจไฟล์ข้อมูล (รันครั้งเดียว) ---
from pathlib import Path
import pandas as pd

DATA_DIR = Path(f'data/week03')
NEEDED = [n for n in ['biz_campaigns.csv', 'biz_customers.csv', 'pb03_M.csv']]

for n in NEEDED:
    df = pd.read_csv(DATA_DIR / n)
    print(f'{n:<22} {df.shape[0]:>5} แถว x {df.shape[1]:>2} คอลัมน์   {list(df.columns)[:6]}{"..." if df.shape[1] > 6 else ""}')

biz_campaigns.csv        240 แถว x  7 คอลัมน์   ['tv', 'radio', 'social', 'email', 'season', 'sales']...
biz_customers.csv       1203 แถว x 13 คอลัมน์   ['gender', 'age', 'income_k', 'tenure_months', 'channel', 'loyalty_member']...
pb03_M.csv                14 แถว x 18 คอลัมน์   ['c0', 'c1', 'c2', 'c3', 'c4', 'c5']...


## ข้อ 3.1 — Null space ของเมทริกซ์ A ขนาด 14×18

ทฤษฎี rank–nullity: **dim(Null A) = n − rank(A)** โดย n = จำนวน**คอลัมน์** = 18
(ไม่ใช่ 14 — เพราะเวกเตอร์ x ยาว 18)

In [3]:
# --- ข้อ 3.1 ---
import numpy as np
import pandas as pd
from pathlib import Path

M = pd.read_csv(Path('data/week03') / 'pb03_M.csv').to_numpy(dtype=float)   # 14x18, แถวแรกคือชื่อคอลัมน์

n_cols = M.shape[1]                          # 18 = ความยาวของเวกเตอร์ x
rank = np.linalg.matrix_rank(M)
answer = n_cols - rank

print('A.shape =', M.shape, '-> n (จำนวนคอลัมน์) =', n_cols)
print('rank(A) =', rank)
print('ค่าที่กรอก: จำนวนเต็ม ->', answer)

A.shape = (14, 18) -> n (จำนวนคอลัมน์) = 18
rank(A) = 9
ค่าที่กรอก: จำนวนเต็ม -> 9


## ข้อ 3.2 — Least Squares: `spend_k` ~ `web_visits` (มี intercept), 891 คนแรก

- X = [1, web_visits], y = spend_k ใช้ `.iloc[:891]` (แถว 0–890)
- คำตอบคือ **β₁ = ความชัน** ไม่ใช่ intercept

In [4]:
# --- ข้อ 3.2 ---
import numpy as np
import pandas as pd
from pathlib import Path

cust = pd.read_csv(Path('data/week03') / 'biz_customers.csv')   # ลูกค้า 1203 คน

D = cust.iloc[:891]                                       # 891 คนแรก
X = np.column_stack([np.ones(891), D['web_visits'].to_numpy(float)])     # [1, web_visits]
y = D['spend_k'].to_numpy(dtype=float)

beta = np.linalg.solve(X.T @ X, X.T @ y)
answer = float(beta[1])

print('X.shape =', X.shape)
print('beta =', np.round(beta, 4), '(b0, b1)')
print('ค่าที่กรอก: 4 ตำแหน่ง ->', f'{answer:.4f}')

X.shape = (891, 2)
beta = [2.484  0.1211] (b0, b1)
ค่าที่กรอก: 4 ตำแหน่ง -> 0.1211


## ข้อ 3.3 — ฉายเวกเตอร์ b ลงบน column space แล้วหาความยาว

- b = `spend_k` ของแถวที่ **557 ถึง 562** (6 คน)
- A = [1, tenure_months] ของ 6 คนเดียวกัน (6×2)
- P = A(AᵀA)⁻¹Aᵀ เป็น **orthogonal projection matrix** (6×6)
- p = Pb คือเวกเตอร์ที่ "ใกล้ b ที่สุดใน column space ของ A"
- คำตอบคือ ‖p‖₂

In [5]:
# --- ข้อ 3.3 ---
import numpy as np
import pandas as pd
from pathlib import Path

cust = pd.read_csv(Path('data/week03') / 'biz_customers.csv')   # ลูกค้า 1203 คน

idx = [557, 558, 559, 560, 561, 562]                    # 6 คน: แถว 557-562
A = np.column_stack([np.ones(6), cust.iloc[idx]['tenure_months'].to_numpy(float)])  # [1, tenure]
b = cust.iloc[idx]['spend_k'].to_numpy(dtype=float)

P = A @ np.linalg.inv(A.T @ A) @ A.T                   # projection matrix 6x6
p = P @ b
answer = float(np.linalg.norm(p, 2))

print('A =', A.T)
print('b =', b)
print('p = Pb =', np.round(p, 4))
print(f'||b|| = {np.linalg.norm(b):.4f} -> ||p|| = {np.linalg.norm(p):.4f} (ฉายแล้วสั้นลงเสมอ)')
print('ค่าที่กรอก: 3 ตำแหน่ง ->', f'{answer:.3f}')

A = [[ 1.  1.  1.  1.  1.  1.]
 [66. 28. 13. 12. 42. 20.]]
b = [6.83 2.81 3.44 2.87 4.41 2.59]
p = Pb = [6.3722 3.671  2.6047 2.5337 4.6662 3.1023]
||b|| = 10.0385 -> ||p|| = 9.9339 (ฉายแล้วสั้นลงเสมอ)
ค่าที่กรอก: 3 ตำแหน่ง -> 9.934


## ข้อ 3.4 — Fit บน 960 คนแรก แล้วพยากรณ์ลูกค้าแถวที่ 1176

- X = [1, tenure_months, basket_size] จาก **960 คนแรกเท่านั้น**
- แถว 1176 ไม่ได้ถูกใช้ในการ fit → เป็นการพยากรณ์ **นอกชุดข้อมูล**
- เวกเตอร์ของแถวใหม่ต้องเรียงคอลัมน์ให้**ตรงลำดับ**กับตอน fit

In [6]:
# --- ข้อ 3.4 ---
import numpy as np
import pandas as pd
from pathlib import Path

cust = pd.read_csv(Path('data/week03') / 'biz_customers.csv')   # ลูกค้า 1203 คน

D = cust.iloc[:960]                                        # 960 คนแรกเท่านั้น
X = np.column_stack([np.ones(960),
                     D['tenure_months'].to_numpy(float),
                     D['basket_size'].to_numpy(float)])
y = D['spend_k'].to_numpy(dtype=float)
beta = np.linalg.solve(X.T @ X, X.T @ y)

new = cust.iloc[1176]                                      # แถวที่ไม่ได้อยู่ในชุด fit
x_new = np.array([1.0, new['tenure_months'], new['basket_size']])   # ลำดับเดียวกัน
answer = float(x_new @ beta)

print('beta =', np.round(beta, 4), '(b0, tenure, basket)')
print('แถว 1176: tenure =', new['tenure_months'], ', basket_size =', new['basket_size'])
print('spend_k จริงของแถวนี้ =', new['spend_k'], '-> ความคลาดเคลื่อน', round(abs(answer - new['spend_k']), 4))
print('ค่าที่กรอก: 3 ตำแหน่ง ->', f'{answer:.3f}')

beta = [2.269  0.0077 0.3099] (b0, tenure, basket)
แถว 1176: tenure = 23 , basket_size = 2.9
spend_k จริงของแถวนี้ = 7.02 -> ความคลาดเคลื่อน 3.6746
ค่าที่กรอก: 3 ตำแหน่ง -> 3.345


## ข้อ 3.5 — ตัวบ่งชี้ไตรมาส Q1 ควบคุมด้วยงบโฆษณาวิทยุ

- design matrix 3 คอลัมน์เรียงตามนี้: (1) ค่าคงที่ 1, (2) ตัวบ่งชี้ Q1, (3) งบวิทยุ
- ตัวบ่งชี้ Q1 = 1 ถ้า `season == 'Q1'` ไม่เช่นนั้น 0
- คำตอบคือ **สัมประสิทธิ์ของตัวบ่งชี้ Q1** = ยอดขายเฉลี่ยของไตรมาสนี้ต่างจากไตรมาสอื่น
  เท่าไรเมื่องบวิทยุเท่ากัน (หน่วย พันชิ้น) — ติดลบแปลว่าไตรมาสนี้ขายน้อยกว่า

In [7]:
# --- ข้อ 3.5 ---
import numpy as np
import pandas as pd
from pathlib import Path

camp = pd.read_csv(Path('data/week03') / 'biz_campaigns.csv')   # แคมเปญ 240 รายการ

X = np.column_stack([
    np.ones(len(camp)),                              # 1) ค่าคงที่
    (camp['season'] == 'Q1').to_numpy(dtype=float),  # 2) ตัวบ่งชี้ Q1
    camp['radio'].to_numpy(dtype=float),            # 3) งบโฆษณาวิทยุ
])
y = camp['sales'].to_numpy(dtype=float)
beta = np.linalg.solve(X.T @ X, X.T @ y)
answer = float(beta[1])

print('จำนวนแคมเปญ Q1 =', int((camp['season'] == 'Q1').sum()), 'จาก', len(camp))
print('beta =', np.round(beta, 4), '(b0, Q1, radio)')
print('ค่าที่กรอก: 4 ตำแหน่ง ->', f'{answer:.4f}')

จำนวนแคมเปญ Q1 = 67 จาก 240
beta = [10.8513 -0.9476  0.3363] (b0, Q1, radio)
ค่าที่กรอก: 4 ตำแหน่ง -> -0.9476


## ข้อ 3.6 — QR decomposition แล้วดู |R[3, 3]|

- X = [1, age, tenure_months, web_visits] ขนาด 1203×4
- `np.linalg.qr(X)` **โหมดเริ่มต้นคือ 'reduced'** → R มีขนาด 4×4 (ไม่ใช่ 1203×4)
- |R[3,3]| คือค่าสมบูรณ์ของสมาชิกแนวทแยงตัวสุดท้าย (ตัวที่เล็กที่สุด)

In [8]:
# --- ข้อ 3.6 ---
import numpy as np
import pandas as pd
from pathlib import Path

cust = pd.read_csv(Path('data/week03') / 'biz_customers.csv')   # ลูกค้า 1203 คน

X = np.column_stack([np.ones(1203),
                     cust['age'].to_numpy(dtype=float),
                     cust['tenure_months'].to_numpy(dtype=float),
                     cust['web_visits'].to_numpy(dtype=float)])      # 1203x4
Q, R = np.linalg.qr(X)                                              # โหมด reduced
answer = float(abs(R[3, 3]))

print('X.shape =', X.shape, '| Q', Q.shape, '| R', R.shape)
print('แนวทแยงของ R =', np.round(np.abs(np.diag(R)), 4))
print('ค่าที่กรอก: 4 ตำแหน่ง ->', f'{answer:.4f}')

X.shape = (1203, 4) | Q (1203, 4) | R (4, 4)
แนวทแยงของ R = [ 34.6843 368.1412 685.1778 115.5478]
ค่าที่กรอก: 4 ตำแหน่ง -> 115.5478


## ข้อ 3.7 — Condition number (2-norm) ของ design matrix

- X = [1, งบวิทยุ, งบโซเชียล, งบอีเมล] ของแคมเปญ 240 รายการ
- **ใช้ค่าดิบ ไม่ standardize**
- cond(X) (2-norm) = σ_max / σ_min
- ยิ่งใหญ่ยิ่ง **ill-conditioned** = ผลของ least squares ไวต่อความคลาดเคลื่อนมาก

In [9]:
# --- ข้อ 3.7 ---
import numpy as np
import pandas as pd
from pathlib import Path

camp = pd.read_csv(Path('data/week03') / 'biz_campaigns.csv')   # แคมเปญ 240 รายการ

X = np.column_stack([np.ones(len(camp)),
                     camp['radio'].to_numpy(dtype=float),
                     camp['social'].to_numpy(dtype=float),
                     camp['email'].to_numpy(dtype=float)])       # 240x4
sv = np.linalg.svd(X, compute_uv=False)                 # ค่าสเงินเฉพาะเรียงจากใหญ่
answer = float(np.linalg.cond(X, 2))

print('X.shape =', X.shape)
print('singular values =', np.round(sv, 4))
print(f'sigma_max / sigma_min = {sv[0]:.4f} / {sv[-1]:.4f}')
print('ค่าที่กรอก: 1 ตำแหน่ง ->', f'{answer:.1f}', '(ยิ่ง > 1e8 ยิ่ง ill-conditioned มาก)')

X.shape = (240, 4)
singular values = [713.8012 241.4854 144.387    4.7334]
sigma_max / sigma_min = 713.8012 / 4.7334
ค่าที่กรอก: 1 ตำแหน่ง -> 150.8 (ยิ่ง > 1e8 ยิ่ง ill-conditioned มาก)


## ข้อ 3.8 — log₁₀ ของ condition number ของ design matrix พหุนามดีกรี 3

- X = [1, age, age², age³] จากลูกค้า 1203 คน
- **ค่าดิบ ไม่ย้ายค่ากลาง ไม่ scale** (นี่คือเหตุผลที่ค่าจะใหญ่มาก เพราะ age³ มีค่าถึง ~343,000
  ทำให้เมทริกซ์เกือบเอียง)
- คำตอบคือ `np.log10(cond(X, 2))`

In [10]:
# --- ข้อ 3.8 ---
import numpy as np
import pandas as pd
from pathlib import Path

cust = pd.read_csv(Path('data/week03') / 'biz_customers.csv')   # ลูกค้า 1203 คน

age = cust['age'].to_numpy(dtype=float)
X = np.column_stack([np.ones(1203), age, age ** 2, age ** 3])     # ไม่ center, ไม่ scale
cond = np.linalg.cond(X, 2)
answer = float(np.log10(cond))

print('ช่วงของคอลัมน์ age^3:', int(age.min()), 'ถึง', int(age.max()))
print(f'cond(X, 2) = {cond:.6g}')
print('ค่าที่กรอก: 2 ตำแหน่ง ->', f'{answer:.2f}')

ช่วงของคอลัมน์ age^3: 18 ถึง 70
cond(X, 2) = 2.66349e+06
ค่าที่กรอก: 2 ตำแหน่ง -> 6.43


## ข้อ 3.9 — จุดยอดของพาราโบลา (fit เฉพาะสมาชิกสะสมแต้ม)

- filter `loyalty_member == 1` (453 คน)
- X = [1, อายุ, อายุ²], y = spend_k
- จุดยอดของพาราโบลา = **−β₁ / (2β₂)**
- ต้องดูเครื่องหมายของ β₂ ด้วย: β₂ > 0 = พาราโบลาหงาย (จุดยอด), β₂ < 0 = พาราโบลาคว่ำ

In [11]:
# --- ข้อ 3.9 ---
import numpy as np
import pandas as pd
from pathlib import Path

cust = pd.read_csv(Path('data/week03') / 'biz_customers.csv')   # ลูกค้า 1203 คน

D = cust[cust.loyalty_member == 1]                       # 453 คน
a = D['age'].to_numpy(dtype=float)
X = np.column_stack([np.ones(len(D)), a, a ** 2])
y = D['spend_k'].to_numpy(dtype=float)
beta = np.linalg.solve(X.T @ X, X.T @ y)
answer = float(-beta[1] / (2 * beta[2]))

print('จำนวนสมาชิกสะสมแต้ม =', len(D), '(โจทย์ระบุ 453)')
print('beta =', np.round(beta, 6), '(b0, b1, b2)')
print('b2 =', round(float(beta[2]), 6), '->', 'พาราโบลาหงาย (จุดยอด)' if beta[2] > 0 else 'พาราโบลาคว่ำ')
print('ค่าที่กรอก: 2 ตำแหน่ง ->', f'{answer:.2f}', 'ปี')

จำนวนสมาชิกสะสมแต้ม = 453 (โจทย์ระบุ 453)
beta = [ 0.761706  0.15522  -0.001766] (b0, b1, b2)
b2 = -0.001766 -> พาราโบลาคว่ำ
ค่าที่กรอก: 2 ตำแหน่ง -> 43.95 ปี


## ข้อ 3.10 — RSS ของการประมาณงบโฆษณารวม

- y = `total_budget` (งบรวมทุกสื่อ) ของแคมเปญ 240 รายการ
- X = [1, งบทีวี, งบโซเชียล, งบอีเมล] (ไม่ใช้ radio เพราะ "ตารางงบวิทยุสูญหาย")
- **RSS = Σ(yᵢ − ŷᵢ)²** ยิ่งเล็กยิ่งดี

In [12]:
# --- ข้อ 3.10 ---
import numpy as np
import pandas as pd
from pathlib import Path

camp = pd.read_csv(Path('data/week03') / 'biz_campaigns.csv')   # แคมเปญ 240 รายการ

X = np.column_stack([np.ones(len(camp)),
                     camp['tv'].to_numpy(dtype=float),
                     camp['social'].to_numpy(dtype=float),
                     camp['email'].to_numpy(dtype=float)])        # 240x4
y = camp['total_budget'].to_numpy(dtype=float)
beta = np.linalg.solve(X.T @ X, X.T @ y)

resid = y - X @ beta
answer = float(resid @ resid)                                  # RSS

print('beta =', np.round(beta, 4), '(b0, tv, social, email)')
print(f'RSE = {np.sqrt(answer / len(y)):.4f} พันบาท')
print('ค่าที่กรอก: 1 ตำแหน่ง ->', f'{answer:.1f}')

beta = [28.2608  0.9939  1.0028  0.9116] (b0, tv, social, email)
RSE = 13.9279 พันบาท
ค่าที่กรอก: 1 ตำแหน่ง -> 46556.8


## สรุปคำตอบทั้ง 10 ข้อ (Week 3)

ค่าทั้งหมดตรวจกับเฉลยในหน้าเว็บแล้ว และอยู่ในเกณฑ์ที่เว็บยอมรับทุกข้อ
วิธีตรวจของเว็บ: `|ค่าที่กรอก - เฉลย| <= ความคลาดเคลื่อน` (ไม่มีการผ่อนปรนเพิ่ม)
| ข้อ | คำอธิบาย | ค่าที่ต้องกรอก | ปัดเศษ | ความคลาดเคลื่อนที่เว็บยอมรับ |
|---|---|---|---|---|
| 3.1 | dim(Null A) ของ A 14x18 | **9** | จำนวนเต็ม | 0.5 |
| 3.2 | beta1 ความชัน spend_k ~ web_visits | **0.1211** | 4 ตำแหน่ง | 0.00015 |
| 3.3 | `‖Pb‖₂` ของการฉายลง column space | **9.934** | 3 ตำแหน่ง | 0.0015 |
| 3.4 | พยากรณ์ spend_k แถว 1176 | **3.345** | 3 ตำแหน่ง | 0.0015 |
| 3.5 | สัมประสิทธิ์ตัวบ่งชี้ Q1 | **-0.9476** | 4 ตำแหน่ง | 0.00015 |
| 3.6 | `R[3,3]` จาก QR decomposition | **115.5478** | 4 ตำแหน่ง | 0.00015 |
| 3.7 | cond(X) 2-norm | **150.8** | 1 ตำแหน่ง | 0.15 |
| 3.8 | log10(cond) ของ [1,a,a^2,a^3] | **6.43** | 2 ตำแหน่ง | 0.015 |
| 3.9 | อายุจุดยอดพาราโบลา (สมาชิกสะสมแต้ม) | **43.95** | 2 ตำแหน่ง | 0.015 |
| 3.10 | RSS ประมาณงบโฆษณารวม | **46556.8** | 1 ตำแหน่ง | 0.15 |

กด **"ส่ง"** ที่หน้าเว็บเพื่อบันทึกคะแนน